# Ekstraksi Kwitansi dengan LightOnOCR-2-1B

Notebook ini membaca kwitansi (penjualan & pembelian) dari file **.zip** yang ada di sebuah folder/directory, mengekstrak datanya dengan model VLM lokal **LightOnOCR-2-1B**, lalu menghitung **Omset** dan **Profit** per nasabah (NIK) per tahun.

Alur:
1. Install & load model
2. Tentukan path folder & nama file zip
3. Extract zip -> loop semua gambar -> ekstrak JSON per kwitansi
4. Gabungkan hasil, hitung omset/profit per NIK per tahun


## 1. Install dependencies

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q pillow


## 2. Import library

In [ ]:
import os
import re
import json
import zipfile
import tempfile
from pathlib import Path

import torch
import pandas as pd
from PIL import Image
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor


## 3. Load model (sama seperti notebook contoh)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16

model = LightOnOcrForConditionalGeneration.from_pretrained(
    "lightonai/LightOnOCR-2-1B", torch_dtype=dtype
).to(device)
processor = LightOnOcrProcessor.from_pretrained("lightonai/LightOnOCR-2-1B")

print(f"Model siap di device: {device}")


## 4. Tentukan lokasi file zip

Ganti `ZIP_DIR` dan `ZIP_FILENAME` sesuai lokasi file zip kwitansi kamu.
Kalau di Google Colab, biasanya `ZIP_DIR = "/content/drive/MyDrive/..."` (setelah mount Drive) atau `"/content"` kalau upload manual.


In [ ]:
# --- UBAH SESUAI KEBUTUHAN ---
ZIP_DIR = "/content"                 # folder tempat file zip berada
ZIP_FILENAME = "kwitansi.zip"        # nama file zip
# ------------------------------

zip_path = Path(ZIP_DIR) / ZIP_FILENAME
assert zip_path.exists(), f"File zip tidak ditemukan: {zip_path}"
print(f"Zip ditemukan: {zip_path}")


### (Opsional) Mount Google Drive kalau file zip ada di Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


## 5. Extract zip ke folder sementara

In [ ]:
tmp_dir = Path(tempfile.mkdtemp(prefix="kwitansi_"))

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(tmp_dir)

image_files = sorted(
    p for p in tmp_dir.rglob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"] and not p.name.startswith(".")
)

print(f"Ditemukan {len(image_files)} kwitansi di dalam {zip_path.name}")
image_files[:5]


## 6. Prompt ekstraksi JSON

In [ ]:
EXTRACTION_PROMPT = """Baca kwitansi di gambar ini dan keluarkan HANYA JSON valid
(tanpa teks lain, tanpa markdown code fence) dengan schema persis:

{
  "nama_usaha": string,
  "nik_pemilik": string,
  "jenis_kwitansi": "penjualan" | "pembelian",
  "no_kwitansi": string,
  "tanggal": "YYYY-MM-DD",
  "pihak_terkait": string,
  "total": number
}

Aturan:
- Nilai "total" HARUS angka murni (integer), buang "Rp" dan pemisah ribuan.
- jenis_kwitansi: "KWITANSI PENJUALAN" -> "penjualan", "KWITANSI PEMBELIAN" -> "pembelian".
- Jika field tidak terbaca, isi null.
- Output HARUS JSON valid, tanpa teks pembuka/penutup, tanpa ```.
"""


## 7. Fungsi ekstraksi satu kwitansi (pola sama seperti notebook contoh: `apply_chat_template` + `model.generate`)

In [ ]:
def extract_one(image_path: Path) -> dict:
    image = Image.open(image_path).convert("RGB")

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": EXTRACTION_PROMPT},
            ],
        }
    ]

    prompt = processor.apply_chat_template(
        conversation, add_generation_prompt=True, tokenize=False
    )

    inputs = processor(text=prompt, images=[image], return_tensors="pt").to(device, dtype=dtype)
    inputs = {
        k: v.to(device=device, dtype=dtype) if v.is_floating_point() else v.to(device)
        for k, v in inputs.items()
    }

    output_ids = model.generate(**inputs, max_new_tokens=512)
    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    output_text = processor.decode(generated_ids, skip_special_tokens=True).strip()

    output_text = output_text.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*\}", output_text, re.DOTALL)
    json_str = match.group(0) if match else output_text

    try:
        parsed = json.loads(json_str)
    except json.JSONDecodeError:
        parsed = {"error": "gagal parse JSON", "raw_output": output_text}

    parsed["source_file"] = image_path.name
    return parsed


## 8. Proses semua kwitansi di dalam zip

In [ ]:
results = []
for i, path in enumerate(image_files, 1):
    print(f"[{i}/{len(image_files)}] Memproses {path.name} ...")
    results.append(extract_one(path))

print("Selesai memproses semua kwitansi.")


## 9. Susun hasil detail per kwitansi

In [ ]:
rows = []
for r in results:
    if "error" in r:
        rows.append(r)
        continue
    rows.append(
        {
            "source_file": r.get("source_file"),
            "no_kwitansi": r.get("no_kwitansi"),
            "jenis_kwitansi": r.get("jenis_kwitansi"),
            "tanggal": r.get("tanggal"),
            "pihak_terkait": r.get("pihak_terkait"),
            "nama_usaha": r.get("nama_usaha"),
            "nik_pemilik": r.get("nik_pemilik"),
            "total": r.get("total"),
        }
    )

df = pd.DataFrame(rows)
df.to_csv("hasil_kwitansi.csv", index=False)
df


## 10. Hitung Omset & Profit per nasabah, per tahun

In [ ]:
df_valid = df[df["total"].notna() & df["tanggal"].notna()].copy() if not df.empty else df

if df_valid.empty:
    print("Tidak ada kwitansi yang berhasil diekstrak dengan lengkap.")
else:
    df_valid["total"] = pd.to_numeric(df_valid["total"], errors="coerce")
    df_valid["tahun"] = pd.to_datetime(df_valid["tanggal"], errors="coerce").dt.year
    df_valid = df_valid[df_valid["tahun"].notna()]
    df_valid["tahun"] = df_valid["tahun"].astype(int)

    grouped = (
        df_valid.groupby(["nik_pemilik", "tahun", "jenis_kwitansi"])["total"]
        .sum()
        .unstack(fill_value=0)
    )
    for col in ["penjualan", "pembelian"]:
        if col not in grouped.columns:
            grouped[col] = 0
    grouped["omset"] = grouped["penjualan"]
    grouped["profit"] = grouped["penjualan"] - grouped["pembelian"]
    grouped = grouped.reset_index()

    nama_usaha_map = df_valid.groupby("nik_pemilik")["nama_usaha"].agg(
        lambda s: s.dropna().iloc[0] if s.notna().any() else None
    )

    summary = grouped.pivot(index="nik_pemilik", columns="tahun", values=["omset", "profit"])
    summary.columns = [f"{metric}_{int(year)}" for metric, year in summary.columns]
    summary = summary.fillna(0).reset_index()
    summary.insert(1, "nama_usaha", summary["nik_pemilik"].map(nama_usaha_map))

    summary.to_csv("ringkasan_omset_profit.csv", index=False)

summary


## 11. Cetak ringkasan

In [ ]:
print("===== OMSET & PROFIT PER NASABAH PER TAHUN =====")
for _, row in summary.iterrows():
    print(f"NIK {row['nik_pemilik']} ({row['nama_usaha']}):")
    for col in summary.columns:
        if col.startswith("omset_") or col.startswith("profit_"):
            print(f"  {col:<12}: Rp {row[col]:,.0f}")

n_errors = sum(1 for r in results if "error" in r)
if n_errors:
    print(f"\nPeringatan: {n_errors} kwitansi gagal di-parse, cek kolom 'error' / 'raw_output' di hasil_kwitansi.csv")
